## Univariate Generation Forecast 

Actual generation forecast depends on multiple factors. Multivariate forecasting is the perfect fit. 

Doing univariate for backup of the website, if we ever need, or start with.

In [1]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])


# Merge plant with generation on 'plant_id'
merged_df = plant.merge(generation_historical, on=['plant_id', 'plant_type' ,'region'], how='left')

df = merged_df[['timestamp', 'plant_id', 'plant_type', 'actual_generation_mw']]

df.head()

,timestamp,plant_id,plant_type,actual_generation_mw
0,2024-01-01 00:00:00,PLANT_001,Solar,0.0
1,2024-01-01 01:00:00,PLANT_001,Solar,0.0
2,2024-01-01 02:00:00,PLANT_001,Solar,0.0
3,2024-01-01 03:00:00,PLANT_001,Solar,0.0
4,2024-01-01 04:00:00,PLANT_001,Solar,0.0


In [ ]:
from src.univariate import prepare_univariate_input, run_univariate_fleet

In [3]:
univ_input = prepare_univariate_input(df)


[Prep] Ready for univariate forecasting:
  Plants included : 50
  Plants skipped  : 0
  Total rows      : 574,850
  Columns         : ['timestamp', 'plant_id', 'plant_type', 'generation']
  Date range      : 2024-01-01 00:00:00 → 2025-04-24 00:00:00

  Rows per plant type:
plant_type
Hybrid     4
Solar     25
Wind      21


In [4]:
# Run — no other changes needed in run_univariate_fleet
forecast_df = run_univariate_fleet(
    fleet_df   = univ_input,
    horizon    = 72,
    n_jobs     = -1,
)


════════════════════════════════════════════════════════════
  FLEET UNIVARIATE FORECAST
  Plants   : 50  (solar=0  wind=0  hybrid=0)
  Horizon  : 72h
  Workers  : 8 / 8 cores  (n_jobs=-1)
  Backend  : loky
  LSTM threshold : 15.0 MAE (hybrid only)
════════════════════════════════════════════════════════════



[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed: 16.6min
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed: 21.0min
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed: 35.3min
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed: 47.2min
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed: 56.5min
[Parallel(n_jobs=8)]: Done  41 out of  50 | elapsed: 65.5min remaining: 14.4min
[Parallel(n_jobs=8)]: Done  47 out of  50 | elapsed: 65.8min remaining:  4.2min
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed: 70.5min finished


  ✓ PLANT_001 (solar) — ets  MAE=1.889
  ✓ PLANT_002 (wind) — theta  MAE=7.616
  ✓ PLANT_003 (hybrid) — prophet  MAE=2.451
  ✓ PLANT_004 (solar) — ets  MAE=2.971
  ✓ PLANT_005 (wind) — ets  MAE=6.252
  ✓ PLANT_006 (solar) — ets  MAE=0.356
  ✓ PLANT_007 (wind) — prophet  MAE=7.45
  ✓ PLANT_008 (solar) — ets  MAE=0.942
  ✓ PLANT_009 (hybrid) — sarima  MAE=3.244
  ✓ PLANT_010 (solar) — theta  MAE=2.711
  ✓ PLANT_011 (wind) — prophet  MAE=7.807
  ✓ PLANT_012 (solar) — theta  MAE=2.934
  ✗ PLANT_013 (hybrid) — ERROR: MemoryError: Unable to allocate 219. MiB for an array with shape (50, 50, 11498) and data type float64
  ✓ PLANT_014 (solar) — ets  MAE=1.382
  ✗ PLANT_015 (wind) — ERROR: MemoryError: Unable to allocate 219. MiB for an array with shape (50, 50, 11497) and data type float64
  ✓ PLANT_016 (solar) — ets  MAE=0.377
  ✓ PLANT_017 (wind) — sarima  MAE=4.715
  ✓ PLANT_018 (solar) — sarima  MAE=2.448
  ✓ PLANT_019 (hybrid) — ets  MAE=1.361
  ✓ PLANT_020 (solar) — theta  MAE=4.151
  ✓ 

## Curtailment Forecasting

curtailment_mw         →   grid policy / demand patterns

In [12]:
from src.curtailment import run_curtailment_fleet, apply_curtailment_correction


# ── Prepare the single input dataframe ───────────────────────────────
hist = merged_df[['timestamp', 'plant_id', 'plant_type', "capacity_mw",'actual_generation_mw','curtailment_mw', 'irradiance_wm2']].copy()
hist["is_forecast_row"] = 0
hist["forecast_mw"]     = hist["actual_generation_mw"]

fc = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\univariate_forecasts.csv", parse_dates=["timestamp"])
fc.drop(columns=['lower_90', 'upper_90', 'model'], inplace=True)  
fc = fc.merge(
    plant[["plant_id", "plant_type", "capacity_mw"]],
    on="plant_id", how="left",
)
fc["is_forecast_row"]      = 1
fc["curtailment_mw"]       = 0.0
fc["actual_generation_mw"] = fc["forecast_mw"]
fc[ 'irradiance_wm2']  = 0.0            # unknown for future hours

curtailment_input_df = pd.concat([hist, fc], ignore_index=True)
curtailment_input_df.head()


,timestamp,plant_id,plant_type,capacity_mw,actual_generation_mw,curtailment_mw,irradiance_wm2,is_forecast_row,forecast_mw
0,2024-01-01 00:00:00,PLANT_001,Solar,102,0.0,0.00,0.0,0,0.0
1,2024-01-01 01:00:00,PLANT_001,Solar,102,0.0,0.00,0.0,0,0.0
2,2024-01-01 02:00:00,PLANT_001,Solar,102,0.0,2.45,0.0,0,0.0
3,2024-01-01 03:00:00,PLANT_001,Solar,102,0.0,0.00,0.0,0,0.0
4,2024-01-01 04:00:00,PLANT_001,Solar,102,0.0,0.00,0.0,0,0.0


In [14]:
from src.curtailment import build_curtailment_features

curt_feat_df = build_curtailment_features(hist_df)
print(f"Rows after build_curtailment_features: {len(curt_feat_df)}")
print(f"Columns: {curt_feat_df.columns.tolist()}")

# Check which columns are causing the dropna wipeout
print("\n=== NULL COUNTS BEFORE dropna ===")
test = hist_df.copy()
test["timestamp"] = pd.to_datetime(test["timestamp"])
test = test.sort_values(["plant_id", "timestamp"]).reset_index(drop=True)

# Reproduce the lag columns
test["curtailed"] = (test["curtailment_mw"] > 0.1).astype(int)
for lag in [1, 2, 3, 24, 48, 168]:
    test[f"curtailed_lag_{lag}"]  = test["curtailed"].shift(lag)
    test[f"curtail_mw_lag_{lag}"] = test["curtailment_mw"].shift(lag)
test["curtail_rate_24h"]  = test["curtailed"].shift(1).rolling(24).mean()
test["curtail_rate_168h"] = test["curtailed"].shift(1).rolling(168).mean()
test["curtail_mean_24h"]  = test["curtailment_mw"].shift(1).rolling(24).mean()

lag_cols = [c for c in test.columns if "lag" in c or "rate" in c or "mean_24h" in c]
print(test[lag_cols].isna().sum().to_string())

print(f"\nRows before dropna : {len(test)}")
print(f"Rows after  dropna : {len(test.dropna())}")

Rows after build_curtailment_features: 574682
Columns: ['timestamp', 'plant_id', 'plant_type', 'capacity_mw', 'actual_generation_mw', 'curtailment_mw', 'irradiance_wm2', 'is_forecast_row', 'forecast_mw', 'curtailed', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak_solar', 'is_monsoon', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'gen_pressure', 'irradiance_norm', 'curtailed_lag_1', 'curtail_mw_lag_1', 'curtailed_lag_2', 'curtail_mw_lag_2', 'curtailed_lag_3', 'curtail_mw_lag_3', 'curtailed_lag_24', 'curtail_mw_lag_24', 'curtailed_lag_48', 'curtail_mw_lag_48', 'curtailed_lag_168', 'curtail_mw_lag_168', 'curtail_rate_24h', 'curtail_rate_168h', 'curtail_mean_24h', 'plant_type_code']

=== NULL COUNTS BEFORE dropna ===
curtailed_lag_1         1
curtail_mw_lag_1        1
curtailed_lag_2         2
curtail_mw_lag_2        2
curtailed_lag_3         3
curtail_mw_lag_3        3
curtailed_lag_24       24
curtail_mw_lag_24      24
curtailed_lag_48       48
curtail_mw_lag_48      48
curt

In [15]:
# ── Run ───────────────────────────────────────────────────────────────
curtailment_forecast_df = run_curtailment_fleet(
    curtailment_input_df = curtailment_input_df,
    val_days             = 14,
    n_jobs               = -1,
)
forecast_df = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\univariate_forecasts.csv", parse_dates=["timestamp"])
final_forecast_df = apply_curtailment_correction(
    generation_forecast_df  = forecast_df,
    curtailment_forecast_df = curtailment_forecast_df,
)

Building curtailment features...
Building future curtailment features...
  [WARN] PLANT_013 — no forecast rows found, skipping
  [WARN] PLANT_015 — no forecast rows found, skipping
  [WARN] PLANT_024 — no forecast rows found, skipping

════════════════════════════════════════════════════════════
  FLEET CURTAILMENT FORECAST
  Plants   : 47
  Workers  : 8 / 8 cores
  Val days : 14
════════════════════════════════════════════════════════════



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    9.8s
[Parallel(n_jobs=-1)]: Done  42 out of  47 | elapsed:   15.4s remaining:    1.7s
[Parallel(n_jobs=-1)]: Done  47 out of  47 | elapsed:   15.6s finished


  ✓ PLANT_001
  ✓ PLANT_002
  ✓ PLANT_003
  ✓ PLANT_004
  ✓ PLANT_005
  ✓ PLANT_006
  ✓ PLANT_007
  ✓ PLANT_008
  ✓ PLANT_009
  ✓ PLANT_010
  ✓ PLANT_011
  ✓ PLANT_012
  ✓ PLANT_014
  ✓ PLANT_016
  ✓ PLANT_017
  ✓ PLANT_018
  ✓ PLANT_019
  ✓ PLANT_020
  ✓ PLANT_021
  ✓ PLANT_022
  ✓ PLANT_023
  ✓ PLANT_025
  ✓ PLANT_026
  ✓ PLANT_027
  ✓ PLANT_028
  ✓ PLANT_029
  ✓ PLANT_030
  ✓ PLANT_031
  ✓ PLANT_032
  ✓ PLANT_033
  ✓ PLANT_034
  ✓ PLANT_035
  ✓ PLANT_036
  ✓ PLANT_037
  ✓ PLANT_038
  ✓ PLANT_039
  ✓ PLANT_040
  ✓ PLANT_041
  ✓ PLANT_042
  ✓ PLANT_043
  ✓ PLANT_044
  ✓ PLANT_045
  ✓ PLANT_046
  ✓ PLANT_047
  ✓ PLANT_048
  ✓ PLANT_049
  ✓ PLANT_050

Curtailment forecasts saved → data\curtailment/curtailment_forecasts.csv

[Curtailment correction applied]
  PLANT_001: gross=1086.7 MWh | curtailment=8.6 MWh (0.8%) | net=1082.1 MWh
  PLANT_002: gross=2774.7 MWh | curtailment=9.6 MWh (0.3%) | net=2765.1 MWh
  PLANT_003: gross=1262.0 MWh | curtailment=7.5 MWh (0.6%) | net=1254.4 MWh
  PLAN

In [16]:
curtailment_forecast_df.head()

,plant_id,timestamp,curtailment_mw_forecast,curtail_probability,curtail_amount_given_event
0,PLANT_001,2025-04-24 01:00:00,0.121,0.078,1.553
1,PLANT_001,2025-04-24 02:00:00,0.121,0.078,1.553
2,PLANT_001,2025-04-24 03:00:00,0.140,0.090,1.553
3,PLANT_001,2025-04-24 04:00:00,0.140,0.090,1.553
4,PLANT_001,2025-04-24 05:00:00,0.140,0.090,1.553


In [17]:
final_forecast_df.head()

,plant_id,timestamp,gross_forecast_mw,lower_90,upper_90,model,curtailment_mw_forecast,curtail_probability,net_forecast_mw
0,PLANT_001,2025-04-24 01:00:00,0.0,0.0,0.0,ets,0.121,0.078,0.0
1,PLANT_001,2025-04-24 02:00:00,0.0,0.0,0.0,ets,0.121,0.078,0.0
2,PLANT_001,2025-04-24 03:00:00,0.0,0.0,0.0,ets,0.140,0.090,0.0
3,PLANT_001,2025-04-24 04:00:00,0.0,0.0,0.0,ets,0.140,0.090,0.0
4,PLANT_001,2025-04-24 05:00:00,0.0,0.0,0.0,ets,0.140,0.090,0.0


In [18]:
curtailment_forecast_df.to_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\curtailment_forecasts.csv", index=False)
final_forecast_df.to_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\final_univariate_forecasts.csv", index=False)

## Irradiance Forecasting 

irradiance             →   solar geometry + cloud patterns  

In [3]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])
generation_historical = generation_historical[['plant_id', 'timestamp', 'irradiance_wm2']]

In [4]:
from src.irradiance import run_irradiance_fleet

# Run both auxiliary forecasts
irradiance_fc = run_irradiance_fleet(generation_historical, plant, n_jobs=-1)


[Irradiance prep] 50 plants | 574,850 rows | daytime mean: 457.5 W/m²

═══════════════════════════════════════════════════════
  IRRADIANCE FLEET FORECAST
  Plants: 50 | Workers: 8 | Horizon: 72h
═══════════════════════════════════════════════════════


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed: 14.4min
[Parallel(n_jobs=8)]: Done  46 out of  50 | elapsed: 82.9min remaining:  7.2min


  ✓ PLANT_001 → prophet
  ✓ PLANT_002 → prophet
  ✓ PLANT_003 → prophet
  ✓ PLANT_004 → prophet
  ✓ PLANT_005 → prophet
  ✓ PLANT_006 → prophet
  ✓ PLANT_007 → prophet
  ✓ PLANT_008 → prophet
  ✓ PLANT_009 → prophet
  ✓ PLANT_010 → prophet
  ✓ PLANT_011 → prophet
  ✓ PLANT_012 → prophet
  ✓ PLANT_013 → prophet
  ✓ PLANT_014 → prophet
  ✓ PLANT_015 → prophet
  ✓ PLANT_016 → prophet
  ✓ PLANT_017 → prophet
  ✓ PLANT_018 → prophet
  ✓ PLANT_019 → prophet
  ✓ PLANT_020 → prophet
  ✓ PLANT_021 → prophet
  ✓ PLANT_022 → prophet
  ✓ PLANT_023 → prophet
  ✓ PLANT_024 → prophet
  ✓ PLANT_025 → prophet
  ✓ PLANT_026 → prophet
  ✓ PLANT_027 → prophet
  ✓ PLANT_028 → prophet
  ✓ PLANT_029 → prophet
  ✓ PLANT_030 → prophet
  ✓ PLANT_031 → prophet
  ✓ PLANT_032 → prophet
  ✓ PLANT_033 → prophet
  ✓ PLANT_034 → prophet
  ✓ PLANT_035 → prophet
  ✓ PLANT_036 → prophet
  ✓ PLANT_037 → prophet
  ✓ PLANT_038 → prophet
  ✓ PLANT_039 → prophet
  ✓ PLANT_040 → prophet
  ✓ PLANT_041 → prophet
  ✓ PLANT_042 → 

[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed: 86.6min finished


## Health Factor Forecasting

In [1]:
import pandas as pd

lifecycle = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\lifecycle_events.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])
generation_historical = generation_historical[['plant_id', 'timestamp', 'health_factor']]

In [2]:
from src.health_factor import run_health_fleet

health_fc  = run_health_fleet(generation_historical, lifecycle, n_jobs=-1)


[Health prep] 50 plants | 574,850 rows | fleet mean health: 0.7869

═══════════════════════════════════════════════════════
  HEALTH FACTOR FLEET FORECAST
  Plants: 50 | Workers: 8 | Horizon: 72h
═══════════════════════════════════════════════════════


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:    9.3s
[Parallel(n_jobs=8)]: Done  46 out of  50 | elapsed:   10.0s remaining:    0.8s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:   10.1s finished


  ✓ PLANT_001 → 0 historical repair events
  ✓ PLANT_002 → 0 historical repair events
  ✓ PLANT_003 → 0 historical repair events
  ✓ PLANT_004 → 0 historical repair events
  ✓ PLANT_005 → 1 historical repair events
  ✓ PLANT_006 → 0 historical repair events
  ✓ PLANT_007 → 0 historical repair events
  ✓ PLANT_008 → 0 historical repair events
  ✓ PLANT_009 → 0 historical repair events
  ✓ PLANT_010 → 0 historical repair events
  ✓ PLANT_011 → 0 historical repair events
  ✓ PLANT_012 → 1 historical repair events
  ✓ PLANT_013 → 1 historical repair events
  ✓ PLANT_014 → 0 historical repair events
  ✓ PLANT_015 → 0 historical repair events
  ✓ PLANT_016 → 0 historical repair events
  ✓ PLANT_017 → 0 historical repair events
  ✓ PLANT_018 → 1 historical repair events
  ✓ PLANT_019 → 1 historical repair events
  ✓ PLANT_020 → 1 historical repair events
  ✓ PLANT_021 → 0 historical repair events
  ✓ PLANT_022 → 0 historical repair events
  ✓ PLANT_023 → 0 historical repair events
  ✓ PLANT_0

## Availability Forecasting

availability_mw        →   outage + maintenance schedules

In [5]:
import pandas as pd

generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])
generation_historical.columns

Index(['timestamp', 'plant_id', 'plant_type', 'region', 'actual_generation_mw',
       'availability_mw', 'curtailment_mw', 'health_factor', 'irradiance_wm2',
       'status'],
      dtype='str')

In [6]:
from src.availability import run_availability_fleet
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])

merged_df = plant.merge(generation_historical, on=['plant_id', 'plant_type' ,'region'], how='left')
merged_df = merged_df[["timestamp", "plant_id", "availability_mw", "capacity_mw"]]

availability_fc  = run_availability_fleet(
    merged_df = merged_df,
    horizon   = 72,
    n_jobs    = -1,
)

[Availability prep] 50 plants | 574,850 rows | avg utilisation: 78.7%

═══════════════════════════════════════════════════════
  AVAILABILITY FLEET FORECAST
  Plants: 50 | Workers: 8 | Horizon: 72h
═══════════════════════════════════════════════════════


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed: 16.0min
[Parallel(n_jobs=8)]: Done  46 out of  50 | elapsed: 173.9min remaining: 15.1min
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed: 177.5min finished


  ✓ PLANT_001 → prophet
  ✓ PLANT_002 → prophet
  ✓ PLANT_003 → prophet
  ✓ PLANT_004 → prophet
  ✓ PLANT_005 → prophet
  ✓ PLANT_006 → prophet
  ✓ PLANT_007 → prophet
  ✓ PLANT_008 → prophet
  ✓ PLANT_009 → prophet
  ✓ PLANT_010 → prophet
  ✓ PLANT_011 → prophet
  ✓ PLANT_012 → prophet
  ✓ PLANT_013 → prophet
  ✓ PLANT_014 → prophet
  ✓ PLANT_015 → prophet
  ✓ PLANT_016 → prophet
  ✓ PLANT_017 → prophet
  ✓ PLANT_018 → prophet
  ✓ PLANT_019 → prophet
  ✓ PLANT_020 → prophet
  ✓ PLANT_021 → prophet
  ✓ PLANT_022 → prophet
  ✓ PLANT_023 → prophet
  ✓ PLANT_024 → prophet
  ✓ PLANT_025 → prophet
  ✓ PLANT_026 → prophet
  ✓ PLANT_027 → prophet
  ✓ PLANT_028 → prophet
  ✓ PLANT_029 → prophet
  ✓ PLANT_030 → prophet
  ✓ PLANT_031 → prophet
  ✓ PLANT_032 → prophet
  ✓ PLANT_033 → prophet
  ✓ PLANT_034 → prophet
  ✓ PLANT_035 → prophet
  ✓ PLANT_036 → prophet
  ✓ PLANT_037 → prophet
  ✓ PLANT_038 → prophet
  ✓ PLANT_039 → prophet
  ✓ PLANT_040 → prophet
  ✓ PLANT_041 → prophet
  ✓ PLANT_042 → 

# Multivariate Generation Forecasting

## Importing Forecast Data 

- Plant
- Weather 
- availability
- curtailment 
- irradiance
- health factor

In [1]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
weather_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\weather_forecast.csv", parse_dates=["timestamp"])
curtailment_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\curtailment_forecasts.csv", parse_dates=["timestamp"])
irradiance_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\irradiance_forecasts.csv", parse_dates=["timestamp"])
health_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\health_forecasts.csv", parse_dates=["timestamp"])
availability_forecast = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\forecasts\availability_forecasts.csv", parse_dates=["timestamp"])

# Merge plant with generation on 'plant_id'
merged_df = plant.merge(curtailment_forecast, on=['plant_id'], how='left')

# Merge with weather on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(weather_forecast, on=['plant_name', 'timestamp'], how='left')

# Merge with irradiance on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(irradiance_forecast, on=['plant_id', 'timestamp'], how='left')

# Merge with health on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(health_forecast, on=['plant_id', 'timestamp'], how='left')

# Merge with availability on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(availability_forecast, on=['plant_id', 'timestamp'], how='left')

forecast_df = merged_df.drop(columns=[ 'plant_name', 'curtail_probability','lower_90',
       'upper_90','lower_90_x', 'upper_90_x','lower_90_y', 'upper_90_y', 'method_x','repair_probability', 'degradation_slope',
    'latitude', 'longitude', 'commission_date', 'region', 'state',
    'developer', 'offtaker','method_y'
])

forecast_df.rename(columns={
    "curtailment_mw_forecast": "curtailment_mw",
    'availability_forecast': "availability_mw",
    "irradiance_forecast": "irradiance_wm2",
    "health_forecast": "health_factor"
}, inplace=True)

forecast_df.columns


Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp', 'curtailment_mw',
       'curtail_amount_given_event', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance', 'irradiance_wm2',
       'health_factor', 'availability_mw'],
      dtype='str')

## Importing Historical Data 

- Plant
- Generation
- Weather

In [2]:
import pandas as pd

plant = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
weather_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\weather_historical.csv", parse_dates=["timestamp"])
generation_historical = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\generation.csv", parse_dates=["timestamp"])


# Merge plant with generation on 'plant_id'
merged_df = plant.merge(generation_historical, on=['plant_id', 'plant_type' ,'region'], how='left')

# Merge with weather on 'plant_id' and 'timestamp'
merged_df = merged_df.merge(weather_historical, on=['plant_name', 'timestamp'], how='left')

historical_df = merged_df.drop(columns=[ 'plant_name', 
    'latitude', 'longitude', 'commission_date', 'region', 'state',
    'developer', 'offtaker','status',
])

historical_df.columns

Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp',
       'actual_generation_mw', 'availability_mw', 'curtailment_mw',
       'health_factor', 'irradiance_wm2', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance'],
      dtype='str')

## Pre-processing

In [3]:
from src.preprocessing import preprocess

historical_df = preprocess(historical_df)
forecast_df = preprocess(forecast_df,is_forecast=True)


────────────────────────────────────────────────────────────
  STARTING PRE-PROCESSING PIPELINE
────────────────────────────────────────────────────────────
[INFO] irradiance_wm2 vs irradiance correlation: 0.6111
[OK] Merged irradiance_wm2 + irradiance → single 'irradiance' column
[OK] Numeric dtypes enforced
[OK] Imputation complete — nulls reduced: 919,880 → 229,940
[WARN] Remaining nulls:
temperature    229940
dtype: int64
[OK] Outlier treatment done (iqr) on: ['actual_generation_mw', 'availability_mw', 'temperature', 'cloud_cover', 'wind_speed', 'irradiance']
[OK] Dtypes optimised — memory usage: 47.1 MB

════════════════════════════════════════════════════════════
  PRE-PROCESSING QA REPORT
════════════════════════════════════════════════════════════
  Shape          : 574,850 rows × 14 columns
  Plants         : 50
  Plant types    : {'Solar': 287425, 'Wind': 241437, 'Hybrid': 45988}
  Date range     : 2024-01-01 00:00:00 → 2025-04-24 00:00:00
  Null values    : 229940
  Duplica

In [4]:
historical_df.columns

Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp',
       'actual_generation_mw', 'availability_mw', 'curtailment_mw',
       'health_factor', 'temperature', 'cloud_cover', 'wind_speed',
       'wind_direction', 'irradiance'],
      dtype='str')

In [5]:
forecast_df.columns

Index(['plant_id', 'plant_type', 'capacity_mw', 'timestamp', 'curtailment_mw',
       'curtail_amount_given_event', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance', 'health_factor',
       'availability_mw'],
      dtype='str')

## Categorize the data

In [4]:
historical_df_solar = historical_df[historical_df['plant_type'] == 'Solar'].copy()
historical_df_wind = historical_df[historical_df['plant_type'] == 'Wind'].copy()
historical_df_hybrid = historical_df[historical_df['plant_type'] == 'Hybrid'].copy()    

forecast_df_solar = forecast_df[forecast_df['plant_type'] == 'Solar'].copy()
forecast_df_wind = forecast_df[forecast_df['plant_type'] == 'Wind'].copy()
forecast_df_hybrid = forecast_df[forecast_df['plant_type'] == 'Hybrid'].copy() 

In [7]:
historical_df_solar[historical_df_solar["plant_id"] == "PLANT_001"].tail()

,plant_id,plant_type,capacity_mw,timestamp,actual_generation_mw,availability_mw,curtailment_mw,health_factor,status,temperature,...,plant_type_Solar,plant_type_Wind,generation,capacity_factor,generation_shortfall_mw,net_availability_mw,health_adjusted_capacity_mw,is_degraded,is_offline,generation_norm
11492,PLANT_001,Solar,102,2025-04-23 20:00:00,0.0,79.900002,0.00,0.783,ON,33.400002,...,1,0,0.0,0.0,79.900002,79.900002,79.865997,0,0,0.0
11493,PLANT_001,Solar,102,2025-04-23 21:00:00,0.0,79.900002,0.00,0.783,ON,32.400002,...,1,0,0.0,0.0,79.900002,79.900002,79.865997,0,0,0.0
11494,PLANT_001,Solar,102,2025-04-23 22:00:00,0.0,79.900002,0.00,0.783,ON,31.200001,...,1,0,0.0,0.0,79.900002,79.900002,79.865997,0,0,0.0
11495,PLANT_001,Solar,102,2025-04-23 23:00:00,0.0,79.900002,1.19,0.783,ON,30.299999,...,1,0,0.0,0.0,79.900002,78.709999,79.865997,0,0,0.0
11496,PLANT_001,Solar,102,2025-04-24 00:00:00,0.0,79.900002,0.00,0.783,ON,30.299999,...,1,0,0.0,0.0,79.900002,79.900002,79.865997,0,0,0.0


## Derived Features

In [5]:

from src.features import build_features

historical_df_solar = build_features(historical_df_solar, "Solar")
historical_df_wind = build_features(historical_df_wind, "Wind")
historical_df_hybrid = build_features(historical_df_hybrid, "Hybrid")

historical_df_solar.drop(columns=['plant_type'], inplace=True)
historical_df_wind.drop(columns=['plant_type'], inplace=True)
historical_df_hybrid.drop(columns=['plant_type'], inplace=True)


forecast_df_solar = build_features(forecast_df_solar, "Solar")
forecast_df_wind = build_features(forecast_df_wind, "Wind")
forecast_df_hybrid = build_features(forecast_df_hybrid, "Hybrid")

forecast_df_solar.drop(columns=['plant_type'], inplace=True)
forecast_df_wind.drop(columns=['plant_type'], inplace=True)
forecast_df_hybrid.drop(columns=['plant_type'], inplace=True)

In [8]:
historical_df_solar.columns

Index(['plant_id', 'capacity_mw', 'timestamp', 'actual_generation_mw',
       'availability_mw', 'curtailment_mw', 'health_factor', 'temperature',
       'cloud_cover', 'wind_speed', 'wind_direction', 'irradiance', 'hour',
       'day_of_year', 'month', 'day_of_week', 'is_weekend', 'hour_sin',
       'hour_cos', 'doy_sin', 'doy_cos', 'month_sin', 'month_cos',
       'clear_sky_irradiance', 'irradiance_adjusted', 'irradiance_ratio',
       'temp_effect', 'expected_generation', 'days_since_cleaning',
       'soiling_loss', 'adjusted_generation_signal', 'is_daylight'],
      dtype='str')

In [7]:
forecast_df_solar.columns

Index(['plant_id', 'capacity_mw', 'timestamp', 'curtailment_mw',
       'curtail_amount_given_event', 'temperature', 'cloud_cover',
       'wind_speed', 'wind_direction', 'irradiance', 'health_factor',
       'availability_mw', 'plant_type_code', 'plant_type_Hybrid',
       'plant_type_Solar', 'plant_type_Wind', 'hour', 'day_of_year', 'month',
       'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'doy_sin',
       'doy_cos', 'month_sin', 'month_cos', 'clear_sky_irradiance',
       'irradiance_adjusted', 'irradiance_ratio', 'temp_effect',
       'expected_generation', 'days_since_cleaning', 'soiling_loss',
       'adjusted_generation_signal', 'is_daylight'],
      dtype='str')

## Forecasting

In [6]:
from src.multivariate import (
    run_multivariate_fleet,   # full fleet: train + forecast + simulate
    run_single_plant,         # one plant: debug / inspect
    run_decomposition_only,   # STL decomposition fleet-wide (no training)
    decompose_series,         # STL for one plant
    simulate_scenarios,       # Monte Carlo standalone
    select_best_multivariate_model,  # train + forecast one plant (returns dict)
    merge_for_multivariate,   # stack historical + forecast rows
)


In [8]:
all_forecasts_df, all_simulations_df = run_multivariate_fleet(
    historical_df = historical_df_solar,
    forecast_df   = forecast_df_solar,
    feature_cols  = None,      # None → auto-select from EXOGENOUS + ENDOGENOUS
    val_days      = 14,        # walk-forward fold size in days
    n_splits      = 3,         # number of walk-forward folds
    n_simulations = 1000,      # Monte Carlo paths per plant
    n_jobs        = -1,        # -1 = use all CPU cores
    output_dir    = "data/multivariate/solar",
)


Merging DataFrames...
[Merge] historical=172,455  forecast=1,080  plants=15
[Merge] Extended: 172,455 historical + 1,080 forecast rows

════════════════════════════════════════════════════════════
  MULTIVARIATE FLEET RUN
  Plants      : 15
  Target      : generation  (MW)
  Horizon     : t+1 .. t+72h
  Val window  : 14 days × 3 folds
  MC paths    : 1000 / plant
  Workers     : 8 / 8 cores
════════════════════════════════════════════════════════════



[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   4 out of  15 | elapsed:  4.2min remaining: 11.5min
[Parallel(n_jobs=8)]: Done   8 out of  15 | elapsed:  4.6min remaining:  4.0min
[Parallel(n_jobs=8)]: Done  12 out of  15 | elapsed:  7.5min remaining:  1.9min
[Parallel(n_jobs=8)]: Done  15 out of  15 | elapsed:  7.7min finished


  ✓ PLANT_001  model=ridge  MAE=0.019
  ✓ PLANT_004  model=ridge  MAE=0.04
  ✓ PLANT_006  model=ridge  MAE=0.003
  ✓ PLANT_008  model=ridge  MAE=0.009
  ✓ PLANT_014  model=ridge  MAE=0.011
  ✓ PLANT_016  model=ridge  MAE=0.003
  ✓ PLANT_018  model=ridge  MAE=0.01
  ✓ PLANT_026  model=ridge  MAE=0.008
  ✓ PLANT_028  model=ridge  MAE=0.028
  ✓ PLANT_030  model=ridge  MAE=0.007
  ✓ PLANT_035  model=ridge  MAE=0.02
  ✓ PLANT_037  model=ridge  MAE=0.015
  ✓ PLANT_039  model=ridge  MAE=0.012
  ✓ PLANT_041  model=ridge  MAE=0.015
  ✓ PLANT_049  model=ridge  MAE=0.014

════════════════════════════════════════════════════════════
  FLEET COMPLETE  15/15 succeeded

  Model selection:
    ridge       : 15 plants (100%)

  Forecast difficulty:
    easy    : 13 plants
    medium  : 2 plants
════════════════════════════════════════════════════════════

  Saved to data\multivariate\solar/


In [10]:
all_forecasts_df, all_simulations_df = run_multivariate_fleet(
    historical_df = historical_df_wind,
    forecast_df   = forecast_df_wind,
    feature_cols  = None,      # None → auto-select from EXOGENOUS + ENDOGENOUS
    val_days      = 14,        # walk-forward fold size in days
    n_splits      = 3,         # number of walk-forward folds
    n_simulations = 1000,      # Monte Carlo paths per plant
    n_jobs        = -1,        # -1 = use all CPU cores
    output_dir    = "data/multivariate/wind",
)


Merging DataFrames...
  [WARN] PLANT_015: in historical_df but no forecast rows
[Merge] historical=149,456  forecast=859  plants=13
[Merge] Extended: 149,456 historical + 859 forecast rows

════════════════════════════════════════════════════════════
  MULTIVARIATE FLEET RUN
  Plants      : 13
  Target      : generation  (MW)
  Horizon     : t+1 .. t+72h
  Val window  : 14 days × 3 folds
  MC paths    : 1000 / plant
  Workers     : 8 / 8 cores
════════════════════════════════════════════════════════════



[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   4 out of  13 | elapsed:  3.8min remaining:  8.6min
[Parallel(n_jobs=8)]: Done   7 out of  13 | elapsed:  4.0min remaining:  3.4min
[Parallel(n_jobs=8)]: Done  10 out of  13 | elapsed:  6.3min remaining:  1.9min
[Parallel(n_jobs=8)]: Done  13 out of  13 | elapsed:  6.5min finished


  ✓ PLANT_002  model=ridge  MAE=0.004
  ✓ PLANT_005  model=ridge  MAE=0.005
  ✓ PLANT_007  model=ridge  MAE=0.005
  ✗ PLANT_015  ValueError: Too few training rows: 0 (need ≥500)
  ✓ PLANT_017  model=ridge  MAE=0.002
  ✓ PLANT_027  model=ridge  MAE=0.007
  ✓ PLANT_029  model=ridge  MAE=0.004
  ✓ PLANT_031  model=ridge  MAE=0.008
  ✓ PLANT_036  model=ridge  MAE=0.008
  ✓ PLANT_038  model=ridge  MAE=0.006
  ✓ PLANT_040  model=ridge  MAE=0.004
  ✓ PLANT_048  model=ridge  MAE=0.004
  ✓ PLANT_050  model=ridge  MAE=0.002

════════════════════════════════════════════════════════════
  FLEET COMPLETE  12/13 succeeded

  Model selection:
    ridge       : 12 plants (100%)

  Forecast difficulty:
    hard    : 12 plants
════════════════════════════════════════════════════════════

  Saved to data\multivariate\wind/


In [11]:
all_forecasts_df, all_simulations_df = run_multivariate_fleet(
    historical_df = historical_df_hybrid,
    forecast_df   = forecast_df_hybrid,
    feature_cols  = None,      # None → auto-select from EXOGENOUS + ENDOGENOUS
    val_days      = 14,        # walk-forward fold size in days
    n_splits      = 3,         # number of walk-forward folds
    n_simulations = 1000,      # Monte Carlo paths per plant
    n_jobs        = -1,        # -1 = use all CPU cores
    output_dir    = "data/multivariate/hybrid",
)


Merging DataFrames...
  [WARN] PLANT_013: in historical_df but no forecast rows
[Merge] historical=22,989  forecast=67  plants=2
[Merge] Extended: 22,989 historical + 67 forecast rows

════════════════════════════════════════════════════════════
  MULTIVARIATE FLEET RUN
  Plants      : 2
  Target      : generation  (MW)
  Horizon     : t+1 .. t+72h
  Val window  : 14 days × 3 folds
  MC paths    : 1000 / plant
  Workers     : 2 / 8 cores
════════════════════════════════════════════════════════════



[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


  ✓ PLANT_003  model=ridge  MAE=0.005
  ✗ PLANT_013  ValueError: Too few training rows: 0 (need ≥500)

════════════════════════════════════════════════════════════
  FLEET COMPLETE  1/2 succeeded

  Model selection:
    ridge       : 1 plants (100%)

  Forecast difficulty:
    easy    : 1 plants
════════════════════════════════════════════════════════════

  Saved to data\multivariate\hybrid/


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:  1.9min finished


In [12]:
stl_summary = run_decomposition_only(
    historical_df=historical_df,
    output_csv="data/multivariate/stl_fleet_summary.csv",
)

stl_summary

[PLANT_001] STL:
  daily seasonal : 0.960  (strong)
  weekly seasonal: 0.957  (strong)
  trend          : 0.022
  residual std   : 3.940 MW
  dominant       : daily  |  difficulty: easy
[PLANT_002] STL:
  daily seasonal : 0.216  (weak)
  weekly seasonal: 0.040  (weak)
  trend          : 0.420
  residual std   : 7.658 MW
  dominant       : daily  |  difficulty: hard
[PLANT_003] STL:
  daily seasonal : 0.909  (strong)
  weekly seasonal: 0.888  (strong)
  trend          : 0.319
  residual std   : 2.779 MW
  dominant       : daily  |  difficulty: easy
[PLANT_004] STL:
  daily seasonal : 0.960  (strong)
  weekly seasonal: 0.957  (strong)
  trend          : 0.020
  residual std   : 5.513 MW
  dominant       : daily  |  difficulty: medium
[PLANT_005] STL:
  daily seasonal : 0.217  (weak)
  weekly seasonal: 0.048  (weak)
  trend          : 0.532
  residual std   : 5.130 MW
  dominant       : daily  |  difficulty: hard
[PLANT_006] STL:
  daily seasonal : 0.952  (strong)
  weekly seasonal: 0.948

,plant_id,seasonal_strength,weekly_strength,trend_strength,residual_std,dominant_period,forecast_difficulty
0,PLANT_038,0.2219,0.0480,0.4380,10.5055,daily,hard
1,PLANT_011,0.2187,0.0467,0.4231,8.0472,daily,hard
2,PLANT_046,0.2097,0.0405,0.4186,8.0464,daily,hard
3,PLANT_029,0.2167,0.0542,0.4257,7.9672,daily,hard
4,PLANT_025,0.2135,0.0476,0.4317,7.7650,daily,hard
5,PLANT_007,0.2115,0.0450,0.4396,7.7291,daily,hard
6,PLANT_002,0.2161,0.0398,0.4200,7.6582,daily,hard
7,PLANT_031,0.2174,0.0403,0.3862,7.6116,daily,hard
8,PLANT_028,0.9586,0.9564,0.0205,7.1935,daily,medium
9,PLANT_021,0.2245,0.0471,0.4351,6.9072,daily,hard


In [ ]:

# Inspect difficulty distribution
print("\nFleet difficulty breakdown:")
print(stl_summary["forecast_difficulty"].value_counts())

# Focus on hard plants that need more attention
hard_plants = stl_summary[stl_summary["forecast_difficulty"] == "hard"]
print(f"\nHard plants ({len(hard_plants)}):")
print(hard_plants[["plant_id", "residual_std", "seasonal_strength"]].to_string(index=False))